# 01 – Baseline: current PII detection on BPMN

This notebook runs the **existing, unchanged** code of
[lblod/pii-identification-anonymization-service](https://github.com/lblod/pii-identification-anonymization-service),
copied into `baseline_service/`, on a synthetic Dutch BPMN corpus (`data/synthetic`, made by `generate_bpmn.py`) and scores it against the known PII.
The numbers here are the baseline that later notebooks try to beat.

Setup: `bash setup.sh`, then `.venv/bin/python generate_bpmn.py`, and pick the kernel *Python (presidio-bpmn-tryout)*.

## 1. Setup: import the baseline service code

In [1]:
from importlib.metadata import version
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

# The service code (identification loads the spaCy models on import)
import baseline_service
from baseline_service.identification import analyzer
from baseline_service.endpoints import anonymize_bpmn as baseline_anonymize_bpmn
from baseline_service.endpoints import identify_bpmn as baseline_identify_bpmn

import pii_eval

print("baseline:", baseline_service.SOURCE_REPO, "@", baseline_service.SOURCE_COMMIT)
for pkg in ["presidio-analyzer", "spacy", "nl-core-news-md", "en-core-web-md"]:
    print(f"{pkg:20} {version(pkg)}")

baseline: https://github.com/lblod/pii-identification-anonymization-service @ d01e7ee
presidio-analyzer    2.2.358
spacy                3.8.16
nl-core-news-md      3.8.0
en-core-web-md       3.8.0


Which recognizers run? `detect_pii` always analyzes with `language="nl"` (the English fallback only triggers on a
`ValueError`). Presidio registers its language-agnostic pattern recognizers for `nl` too; the country-specific ones
(US SSN, UK NHS, credit cards, …) are English-only. The custom Belgian rijksregisternummer recognizer shows up as
`PatternRecognizer`.

In [2]:
def recognizer_table(lang):
    return pd.DataFrame(
        [{"recognizer": r.name, "entities": ", ".join(r.supported_entities)} for r in analyzer.get_recognizers(language=lang)]
    ).sort_values("recognizer").reset_index(drop=True)

recognizer_table("nl")

,recognizer,entities
0,CryptoRecognizer,CRYPTO
1,DateRecognizer,DATE_TIME
2,EmailRecognizer,EMAIL_ADDRESS
3,IbanRecognizer,IBAN_CODE
4,InVoterRecognizer,IN_VOTER
5,IpRecognizer,IP_ADDRESS
6,MedicalLicenseRecognizer,MEDICAL_LICENSE
7,PatternRecognizer,BELGIAN_NATIONAL_REGISTRY
8,PhoneRecognizer,PHONE_NUMBER
9,SpacyRecognizer,"NRP, PHONE_NUMBER, AGE, LOCATION, DATE_TIME, ID, EMAIL, PERSON"


`baseline_service.endpoints` holds the BPMN endpoint logic of the service's `web.py` without Flask:
`identify_bpmn(xml)` (= `POST /bpmn/identify`) and `anonymize_bpmn(xml, pointers)` (= `POST /bpmn/anonymize`).

## 2. The synthetic corpus

In [3]:
files, elements, spans = pii_eval.load_corpus("data/synthetic")
print(f"{len(files)} BPMN files, {len(elements)} text elements, {len(spans)} PII spans "
      f"in {(elements.n_pii > 0).sum()} elements")
pd.crosstab(spans.entity, spans.location, margins=True, margins_name="total")

20 BPMN files, 551 text elements, 205 PII spans in 160 elements


location,documentation,name,text,total
entity,,,,
ADDRESS,6,4,4,14
DATE_OF_BIRTH,3,5,0,8
EMAIL,5,10,5,20
IBAN,3,5,4,12
KBO,3,5,0,8
LICENSE_PLATE,3,5,0,8
PERSON,25,57,13,95
PHONE,8,9,4,21
RRN,6,9,4,19


Where does text sit in the files, and how much of it holds PII?

In [4]:
elements.assign(has_pii=elements.n_pii > 0).groupby("location").agg(
    elements=("text", "size"), with_pii=("has_pii", "sum"), pii_spans=("n_pii", "sum")
)

,elements,with_pii,pii_spans
location,,,
documentation,52,31,62
name,437,99,109
text,62,30,34


In [5]:
# A few examples per location
elements[elements.n_pii > 0].groupby("location").head(4)[["file", "location", "element_type", "text"]]

,file,location,element_type,text
1,aanvraag-rijbewijs-sintniklaas.bpmn,documentation,participant,Bij afwezigheid van Marleen Van de Velde neemt Hanne Jacobs het dossier over.
2,aanvraag-rijbewijs-sintniklaas.bpmn,name,participant,Jan Maes
4,aanvraag-rijbewijs-sintniklaas.bpmn,name,lane,Marleen - Burgerzaken
5,aanvraag-rijbewijs-sintniklaas.bpmn,name,lane,Marleen Van de Velde
7,aanvraag-rijbewijs-sintniklaas.bpmn,name,userTask,Advies vragen aan Jan Maes
23,aanvraag-rijbewijs-sintniklaas.bpmn,text,textAnnotation,Back-up: jan.maes@skynet.be
25,aanvraag-rijbewijs-sintniklaas.bpmn,text,textAnnotation,INSZ 670620 907 39
26,aanvraag-rijbewijs-sintniklaas.bpmn,text,textAnnotation,Rekeningnummer BE50 2852 6085 1318
29,fuifkwaliteitslabel-aalter.bpmn,documentation,participant,"Aanvrager Nele Vandenberghe, wonende te Veldstraat 153, 3500 Hasselt, geboren op 19-08-1960, rijksregisternummer 600..."
39,fuifkwaliteitslabel-aalter.bpmn,documentation,manualTask,"Testdossier: INSZ 600819 817 26, adres Veldstraat 153, 3500 Hasselt."


## 3. Run the baseline

In [6]:
%%time
baseline_outputs = {f: baseline_identify_bpmn(xml) for f, xml in files.items()}

detections = pd.DataFrame([
    {"file": f, "element_id": d["id"], "location": "name", **{k: v for k, v in d.items() if k != "id"}}
    for f, out in baseline_outputs.items() for d in out["pii_results"]
])
print(f"{len(detections)} detections, {sum(len(o['extracted_elements']) for o in baseline_outputs.values())} name attributes scanned")
detections.entity.value_counts()

164 detections, 437 name attributes scanned
CPU times: user 1.7 s, sys: 14.5 ms, total: 1.72 s
Wall time: -6.2e+08 ns


entity
PERSON                       69
LOCATION                     40
DATE_TIME                    32
EMAIL_ADDRESS                10
IBAN_CODE                     5
NRP                           4
BELGIAN_NATIONAL_REGISTRY     4
Name: count, dtype: int64

## 4. Evaluation

In [7]:
scored_spans, scored_det = pii_eval.match(spans, detections)
pii_eval.summarize(scored_spans, scored_det)

,n_pii,n_detections,false_positives,precision,recall,recall_full,recall_typed,f1
scope,,,,,,,,
all,205,164,62,0.622,0.463,0.385,0.4,0.531


- **recall**: share of PII spans touched by a detection; **recall_full**: share completely covered (what
  masking needs, e.g. `Lander Lambrecht` with only `Lambrecht` detected is touched but not covered);
  **recall_typed**: touched by a detection of the right type.
- **precision**: share of detections that touch real PII.

### Per location

In [8]:
pii_eval.summarize(scored_spans, scored_det, by="location")

,n_pii,n_detections,false_positives,precision,recall,recall_full,recall_typed,f1
location,,,,,,,,
documentation,62,0,0,NaN,0.000,0.000,0.000,NaN
name,109,164,62,0.622,0.872,0.725,0.752,0.726
text,34,0,0,NaN,0.000,0.000,0.000,NaN


The service only reads `name` attributes, so PII in `<bpmn:documentation>` and text annotations (`<bpmn:text>`)
is never found.

### Per entity type

In [9]:
pii_eval.summarize(scored_spans, scored_det, by="entity")

,n_pii,recall,recall_full,recall_typed
entity,,,,
ADDRESS,14,0.286,0.000,0.286
DATE_OF_BIRTH,8,0.625,0.625,0.625
EMAIL,20,0.500,0.500,0.500
IBAN,12,0.417,0.417,0.417
KBO,8,0.000,0.000,0.000
LICENSE_PLATE,8,0.000,0.000,0.000
PERSON,95,0.568,0.537,0.568
PHONE,21,0.381,0.143,0.000
RRN,19,0.474,0.263,0.211


Same, restricted to `name` attributes (the part the service does scan):

In [10]:
in_names = scored_spans.location == "name"
pii_eval.summarize(scored_spans[in_names], scored_det, by="entity")

,n_pii,recall,recall_full,recall_typed
entity,,,,
ADDRESS,4,1.000,0.000,1.000
DATE_OF_BIRTH,5,1.000,1.000,1.000
EMAIL,10,1.000,1.000,1.000
IBAN,5,1.000,1.000,1.000
KBO,5,0.000,0.000,0.000
LICENSE_PLATE,5,0.000,0.000,0.000
PERSON,57,0.947,0.895,0.947
PHONE,9,0.889,0.333,0.000
RRN,9,1.000,0.556,0.444


### Missed or partially covered PII in `name` attributes

In [11]:
scored_spans[in_names & ~scored_spans.full][["entity", "value", "hit", "text"]].sort_values("entity")

,entity,value,hit,text
133,ADDRESS,"Dorpsstraat 72 bus 2, 8800 Roeselare",True,"Plaatsbezoek Dorpsstraat 72 bus 2, 8800 Roeselare"
170,ADDRESS,"Stationsstraat 101A, 8800 Roeselare",True,"Plaatsbezoek Stationsstraat 101A, 8800 Roeselare"
31,ADDRESS,"Brugsesteenweg 32, 3000 Leuven",True,"Plaatsbezoek Brugsesteenweg 32, 3000 Leuven"
95,ADDRESS,"Sint-Jansstraat 157, 9300 Aalst",True,"Plaatsbezoek Sint-Jansstraat 157, 9300 Aalst"
164,KBO,0732.659.608,False,Facturatie KBO 0732.659.608
111,KBO,0549.936.451,False,Facturatie KBO 0549.936.451
62,KBO,0433.335.127,False,Facturatie KBO 0433.335.127
200,KBO,0775.681.680,False,Facturatie KBO 0775.681.680
41,KBO,0992.071.062,False,Facturatie KBO 0992.071.062
78,LICENSE_PLATE,2-ZAR-193,False,Nummerplaat 2-ZAR-193 registreren


For context, what did the baseline return on those elements?

In [12]:
partial = scored_spans[in_names & scored_spans.hit & ~scored_spans.full]
partial[pii_eval.KEY].drop_duplicates().merge(scored_det, on=pii_eval.KEY)[["entity", "match", "score", "gt_entity"]]

,entity,match,score,gt_entity
0,PERSON,Van de Velde,0.85,PERSON
1,DATE_TIME,220 20,0.85,RRN
2,PERSON,Plaatsbezoek,0.85,None
3,LOCATION,Leuven,0.85,ADDRESS
4,PERSON,Annelies,0.85,PERSON
5,DATE_TIME,0460,0.85,PHONE
6,DATE_TIME,37 84 19,0.85,PHONE
7,DATE_TIME,0485,0.85,PHONE
8,DATE_TIME,57 97 90,0.85,PHONE
9,DATE_TIME,205 72,0.85,RRN


### False positives

In [13]:
fp = scored_det[~scored_det.tp].merge(elements[pii_eval.KEY + ["text"]], on=pii_eval.KEY)
print(f"{len(fp)} false positives")
fp[["entity", "match", "score", "text"]].sort_values("entity")

62 false positives


,entity,match,score,text
27,DATE_TIME,1 maand,0.85,1 maand voor de fuif
31,DATE_TIME,1 maand,0.85,1 maand voor de fuif
33,DATE_TIME,Aangifte,0.85,Aangifte huwelijk Genk
21,DATE_TIME,31 augustus,0.85,Uiterlijk 31 augustus
19,DATE_TIME,3 maanden,0.85,Aanvraag evenement minstens 3 maanden vooraf
16,DATE_TIME,3 maanden,0.85,Aanvraag evenement minstens 3 maanden vooraf
38,DATE_TIME,Aangifte,0.85,Aangifte huwelijk Hasselt
40,DATE_TIME,31 augustus,0.85,Uiterlijk 31 augustus
28,DATE_TIME,30 dagen,0.85,Binnen 30 dagen
34,LOCATION,Genk,0.85,Aangifte huwelijk Genk


### Detections with the wrong type

In [14]:
scored_det[scored_det.tp & ~scored_det.tp_typed][["entity", "gt_entity", "match", "score"]]

,entity,gt_entity,match,score
13,DATE_TIME,PHONE,+32493276528,0.85
14,DATE_TIME,ADDRESS,157,0.85
28,DATE_TIME,RRN,220 20,0.85
33,DATE_TIME,PHONE,016/42.81.31,0.85
40,DATE_TIME,PHONE,633,0.85
53,DATE_TIME,RRN,205 72,0.85
66,DATE_TIME,ADDRESS,101A,0.85
70,DATE_TIME,PHONE,57 93 27 66,0.85
85,DATE_TIME,RRN,62052837921,0.85
86,DATE_TIME,RRN,630302,0.85


## 5. Anonymization round trip

Feed every baseline detection back into the `/bpmn/anonymize` logic, then check per PII span what is left of it
in the anonymized element text: **leaked** = at least one word or number of the value survives
(e.g. `Lander *****`), **leaked_intact** = the whole value survives.

In [15]:
anonymized = {f: baseline_anonymize_bpmn(xml, baseline_outputs[f]["pii_results"]) for f, xml in files.items()}
residual = pii_eval.residual_pii(spans, anonymized)
print(f"PII (partly) left after anonymization: {residual.leaked.sum()} / {len(residual)} ({residual.leaked.mean():.0%}), "
      f"fully intact: {residual.leaked_intact.sum()}")
residual.groupby("location")[["leaked", "leaked_intact"]].agg(["sum", "mean"]).round(3)

PII (partly) left after anonymization: 123 / 205 (60%), fully intact: 110


leaked        leaked_intact       
                 sum   mean           sum   mean
location                                        
documentation     62  1.000            62  1.000
name              27  0.248            14  0.128
text              34  1.000            34  1.000

In [16]:
example = "fuifkwaliteitslabel-aalter.bpmn"
before = elements[elements.file == example][pii_eval.KEY + ["text"]]
root = pii_eval.ET.fromstring(anonymized[example])
before.assign(after=[pii_eval.element_text(root, e, loc) for e, loc in zip(before.element_id, before.location)]) \
      .loc[lambda d: d.text != d.after, ["location", "text", "after"]]

,location,text,after
28,name,Jeugd - Fuifkwaliteitslabel aanvraag Aalter,Jeugd - ***** aanvraag *****
30,name,Elke Van de Velde,Elke *****
35,name,1 maand voor de fuif,***** voor de fuif
36,name,Terugbetaling op BE32 5678 9073 9102,Terugbetaling op *****
41,name,Controle RRN 60121670958,Controle RRN *****
42,name,Binnen 30 dagen,Binnen *****


PII in that file that is (partly) left in the output:

In [17]:
residual[(residual.file == example) & residual.leaked][["location", "entity", "value", "leaked_intact", "text"]]

,location,entity,value,leaked_intact,text
11,documentation,PERSON,Nele Vandenberghe,True,"Aanvrager Nele Vandenberghe, wonende te Veldstraat 153, 3500 Hasselt, geboren op 19-08-1960, rijksregisternummer 600..."
12,documentation,ADDRESS,"Veldstraat 153, 3500 Hasselt",True,"Aanvrager Nele Vandenberghe, wonende te Veldstraat 153, 3500 Hasselt, geboren op 19-08-1960, rijksregisternummer 600..."
13,documentation,DATE_OF_BIRTH,19-08-1960,True,"Aanvrager Nele Vandenberghe, wonende te Veldstraat 153, 3500 Hasselt, geboren op 19-08-1960, rijksregisternummer 600..."
14,documentation,RRN,600819 817 26,True,"Aanvrager Nele Vandenberghe, wonende te Veldstraat 153, 3500 Hasselt, geboren op 19-08-1960, rijksregisternummer 600..."
15,name,PERSON,Elke Van de Velde,False,Elke Van de Velde
17,documentation,RRN,600819 817 26,True,"Testdossier: INSZ 600819 817 26, adres Veldstraat 153, 3500 Hasselt."
18,documentation,ADDRESS,"Veldstraat 153, 3500 Hasselt",True,"Testdossier: INSZ 600819 817 26, adres Veldstraat 153, 3500 Hasselt."
20,text,PERSON,Nele Vandenberghe,True,Telefoon Nele Vandenberghe: +32 480 33 72 37
21,text,PHONE,+32 480 33 72 37,True,Telefoon Nele Vandenberghe: +32 480 33 72 37
22,text,ADDRESS,"Grote Markt 149, 9100 Sint-Niklaas",True,"Adres: Grote Markt 149, 9100 Sint-Niklaas"


The serialized XML loses the original namespace prefixes (`bpmn:` becomes `ns0:`):

In [18]:
print(files[example][:300])
print("...")
print(anonymized[example][:300])

<?xml version='1.0' encoding='UTF-8'?>
<bpmn:definitions xmlns:bpmn="http://www.omg.org/spec/BPMN/20100524/MODEL" id="Definitions_3rtqohm" targetNamespace="http://bpmn.io/schema/bpmn" exporter="Camunda Modeler" exporterVersion="5.14.0">
  <bpmn:collaboration id="Collaboration_r7sgmso">
    <bpmn:par
...
<?xml version='1.0' encoding='utf-8'?>
<ns0:definitions xmlns:ns0="http://www.omg.org/spec/BPMN/20100524/MODEL" id="Definitions_3rtqohm" targetNamespace="http://bpmn.io/schema/bpmn" exporter="Camunda Modeler" exporterVersion="5.14.0">
  <ns0:collaboration id="Collaboration_r7sgmso">
    <ns0:partici


## 6. Baseline summary

In [19]:
summary = pd.concat([
    pii_eval.summarize(scored_spans, scored_det).assign(group="overall"),
    pii_eval.summarize(scored_spans, scored_det, by="location").assign(group="location"),
    pii_eval.summarize(scored_spans, scored_det, by="entity").assign(group="entity"),
]).rename_axis("scope").reset_index()
summary.insert(0, "approach", "baseline")
summary["residual_pii"] = summary["scope"].map(
    {"all": residual.leaked.mean(), **residual.groupby("location").leaked.mean(), **residual.groupby("entity").leaked.mean()}
).round(3)

Path("results").mkdir(exist_ok=True)
summary.to_csv("results/baseline.csv", index=False)
summary

,approach,scope,n_pii,n_detections,false_positives,precision,recall,recall_full,recall_typed,f1,group,residual_pii
0,baseline,all,205,164.0,62.0,0.622,0.463,0.385,0.400,0.531,overall,0.600
1,baseline,documentation,62,0.0,0.0,NaN,0.000,0.000,0.000,NaN,location,1.000
2,baseline,name,109,164.0,62.0,0.622,0.872,0.725,0.752,0.726,location,0.248
3,baseline,text,34,0.0,0.0,NaN,0.000,0.000,0.000,NaN,location,1.000
4,baseline,ADDRESS,14,NaN,NaN,NaN,0.286,0.000,0.286,NaN,entity,1.000
5,baseline,DATE_OF_BIRTH,8,NaN,NaN,NaN,0.625,0.625,0.625,NaN,entity,0.375
6,baseline,EMAIL,20,NaN,NaN,NaN,0.500,0.500,0.500,NaN,entity,0.500
7,baseline,IBAN,12,NaN,NaN,NaN,0.417,0.417,0.417,NaN,entity,0.583
8,baseline,KBO,8,NaN,NaN,NaN,0.000,0.000,0.000,NaN,entity,1.000
9,baseline,LICENSE_PLATE,8,NaN,NaN,NaN,0.000,0.000,0.000,NaN,entity,1.000


### Where to improve next

The tables above point at the main gaps; candidate steps for the next notebooks:

1. **Coverage**: also scan `<bpmn:documentation>` and `<bpmn:textAnnotation>/<bpmn:text>` (and keep the anonymizer
   writing back to those locations, with the original namespace prefixes).
2. **Belgian formats**: phone numbers in Belgian notation, rijksregisternummer written with spaces, KBO numbers,
   license plates, addresses (street + number + postcode + municipality).
3. **Precision**: the 11-digit RRN regex fires on case numbers; validate the mod-97 checksum and use context words.
4. **Names**: partial name spans (first name missed) and short first-name-only mentions.